   
# 03 – Clean Stops

This notebook implements an **SCD Type 2** pipeline for transit stop data. It reads a bronze stops table, ensures the silver SCD2 target table exists, then merges records — expiring changed rows and inserting new/updated ones as current.

In [0]:
import os
import sys
sys.path.insert(0, '../exports')
from cleaning_functions_export import *

## Inspect Input Parameters

In [0]:
bronze_stops_table = dbutils.widgets.get("bronze_stops_table")
silver_stops_scd2_table = dbutils.widgets.get("silver_stops_scd2_table")
print(f"bronze_stops_table = {bronze_stops_table}")
print(f"silver_stops_scd2_table = {silver_stops_scd2_table}")

## Merge Data into SCD2 Table

In [0]:
ensure_silver_stops_table(spark, silver_stops_scd2_table)

In [0]:
expired, inserted = apply_stops_scd2(spark, bronze_stops_table, silver_stops_scd2_table)
print(f"Expired: {expired} | Inserted: {inserted}")

Expired: 0 | Inserted: 0


In [0]:
%sql
SELECT
  COUNT(*)             AS total_rows,
  SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows,
  SUM(CASE WHEN NOT is_current THEN 1 ELSE 0 END) AS expired_rows,
  MIN(valid_from)      AS earliest_valid_from,
  MAX(valid_from)      AS latest_valid_from
FROM IDENTIFIER(:silver_stops_scd2_table)

total_rows,current_rows,expired_rows,earliest_valid_from,latest_valid_from
8357,8357,0,2026-04-19T03:50:09.000Z,2026-04-19T03:50:09.000Z
